# MAF-OOD v5.2 seed42 dual-track

この notebook は `MAF_OOD_v51.ipynb` を基準に、seed=42 での再実験用に整理し直した版です。

- `Track I`: 各手法の **公式 repo / 論文準拠** の外部実行結果を CSV として取り込みます
- `Track II`: 元 notebook の `Proj(p=128)` を共有表現として使う **同バックボーン・同一射影次元** 評価を notebook 内で計算します
- `MSP / MaxLogit / Energy / Entropy` は `II` のみ
- `GEN / KNN / RMD / NCM Agreement / ViM / OODD` は `I / II` を両方並べますが、`I` は外部 CSV を import したときだけ最終表に入ります
- `MAF` は提案手法として、ID validation だけで fit した adaptive alpha を最終ランキングに載せます

注意:
- `Track I` は notebook がローカル近似で勝手に埋めません。厳密にやるために外部の公式実行結果 CSV を受け取ります
- `Track II` の logit は、128 次元 projection から raw logit を近似する線形 readout を学習して作ります
- この notebook は `maf_ood_notebook_utils.py` を参照します


In [ ]:
# Dependency check
import importlib

required = [
    "numpy",
    "scipy",
    "sklearn",
    "torch",
    "torchvision",
    "timm",
    "open_clip",
    "pandas",
    "matplotlib",
    "seaborn",
]

missing = []
for name in required:
    try:
        importlib.import_module(name)
    except Exception as exc:
        missing.append((name, repr(exc)))

try:
    import lzma
except Exception as exc:
    missing.append(("lzma", repr(exc)))

if missing:
    print("Missing or broken dependencies:")
    for name, exc in missing:
        print(f"- {name}: {exc}")
    raise RuntimeError("Fix the dependencies above before running the remaining cells.")

print("Dependency check passed.")


In [ ]:
# Config
from pathlib import Path
import torch

SAVE_ROOT = str(Path("~/maf_ood_v51").expanduser())
DATA_SRC = str(Path("~/WILD_DATA/splits").expanduser())
BACKBONES = ["imagenet_vit", "bioclip"]
SEED = 42
FORCE_REEXTRACT = False
EVAL_ONLY = False
OFFICIAL_TRACK_I_CSV = None  # 例: str(Path("~/maf_ood_v51/official_track_i_results.csv").expanduser())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"SAVE_ROOT: {SAVE_ROOT}")
print(f"DATA_SRC  : {DATA_SRC}")
print(f"BACKBONES : {BACKBONES}")
print(f"SEED      : {SEED}")
print(f"Track I CSV: {OFFICIAL_TRACK_I_CSV}")
print(f"DEVICE    : {DEVICE}")
print("Track II uses the shared Proj(p=128) from MAF_OOD_v51.ipynb")


In [ ]:
# Imports
import pandas as pd
import maf_ood_notebook_utils as nb

print(f"Shared projection dim: {nb.COMMON_PROJ_DIM}")
print(f"Alpha sweep points   : {len(nb.ALPHA_SWEEP)}")


## Source manifest

ここで使う `Track I` の基準は、ユーザー指定の原典実装・論文に合わせた **外部実行結果** です。`Track II` は元 notebook の 128 次元 projection を共有表現にする fair track です。


In [ ]:
nb.source_manifest()

In [ ]:
# Run seed=42 for each backbone
cfg = nb.Cfg()
all_results = {}

for backbone in BACKBONES:
    print(f"\n{'=' * 120}")
    print(f"Running: {backbone} / seed={SEED}")
    print(f"{'=' * 120}")
    result = nb.evaluate_backbone_seed42(
        backbone=backbone,
        data_src=DATA_SRC,
        save_root=SAVE_ROOT,
        cfg=cfg,
        device=DEVICE,
        force_reextract=FORCE_REEXTRACT,
        eval_only=EVAL_ONLY,
        official_track_i_csv=OFFICIAL_TRACK_I_CSV,
    )
    artifact_paths = nb.save_backbone_artifacts(result)
    all_results[backbone] = {
        "result": result,
        "artifacts": artifact_paths,
    }
    print("Saved artifacts:")
    if result["official_track_i"]["imported"]:
        print(f"Imported official Track I rows from: {result['official_track_i']['csv']}")
        print(f"Methods: {', '.join(result['official_track_i']['methods'])}")
    for key, value in artifact_paths.items():
        print(f"- {key}: {value}")


## Ranking tables

最終ランキングは backbone ごとに表示します。`MAF` は ID validation から決めた adaptive alpha の 1 行です。固定 alpha sweep は分析用として別に残します。`OODD` など backbone 外条件の official Track I があれば最後に別枠で表示します。


In [ ]:
for backbone in BACKBONES:
    print(f"\n### {backbone}")
    display(all_results[backbone]["result"]["results"])

if OFFICIAL_TRACK_I_CSV:
    imported = nb.load_external_track_i_csv(OFFICIAL_TRACK_I_CSV)
    extra = imported[~imported["backbone"].isin(BACKBONES)].copy()
    if not extra.empty:
        print("\n### official-only Track I conditions")
        display(extra.sort_values(["AUROC", "FPR95"], ascending=[False, True]))


In [ ]:
for backbone in BACKBONES:
    result = all_results[backbone]["result"]
    nb.plot_method_ranking(result["results"], title=f"{backbone} ranking", top_k=None)
    nb.plot_top_roc_curves(result, top_k=5, title_prefix=backbone)


## ID classification quality

`raw head` は現在の学習済み分類 head、`projected readout` は `Track II` 用の 128-d readout です。


In [ ]:
for backbone in BACKBONES:
    result = all_results[backbone]["result"]

    print(f"\n### {backbone} / raw head")
    display(result["raw_accuracy"])
    nb.plot_class_accuracy(result["raw_accuracy"], title=f"{backbone} raw-head class accuracy")
    nb.plot_confusion(result["raw_confusion"], nb.ID_CLASSES, title=f"{backbone} raw-head confusion")

    print(f"\n### {backbone} / projected readout")
    display(result["proj_accuracy"])
    nb.plot_class_accuracy(result["proj_accuracy"], title=f"{backbone} projected-readout class accuracy")
    nb.plot_confusion(result["proj_confusion"], nb.ID_CLASSES, title=f"{backbone} projected-readout confusion")


## MAF alpha sweep

要求どおり、`alpha=0.00..1.00` を `0.01` 刻みで全て出します。


In [ ]:
for backbone in BACKBONES:
    alpha_df = all_results[backbone]["result"]["alpha_sweep"].copy()
    print(f"\n### {backbone}")
    display(alpha_df[["alpha", "AUROC", "AUPR-IN", "AUPR-OUT", "FPR95", "AUTC"]])
    nb.plot_maf_alpha_sweep(alpha_df, title=f"{backbone} / MAF alpha sweep")


In [ ]:
# Combined summary across backbones
combined_results = pd.concat(
    [all_results[bb]["result"]["results"] for bb in BACKBONES],
    ignore_index=True,
)
combined_alpha = pd.concat(
    [all_results[bb]["result"]["alpha_sweep"] for bb in BACKBONES],
    ignore_index=True,
)

summary_root = Path(SAVE_ROOT).expanduser() / "seed42_dualtrack_summary"
summary_root.mkdir(parents=True, exist_ok=True)
combined_results_path = summary_root / "combined_results_seed42.csv"
combined_alpha_path = summary_root / "combined_alpha_sweep_seed42.csv"
combined_results.to_csv(combined_results_path, index=False)
combined_alpha.to_csv(combined_alpha_path, index=False)

print(combined_results_path)
print(combined_alpha_path)
display(combined_results)


## Notes

- `Track I` と `Track II` の差は、**score 関数の式だけでなく、使う表現空間** にあります
- `NCM Agreement` は score ルール自体は変えず、`raw` と `projection` の空間差だけを見ます
- `GEN` は公式 `benchmark.py` に合わせた `gamma=0.1` を使います
- `OODD` は公式 repo の ImageNet 側 config (`K1=100, K2=5, alpha=0.5, queue=2048`) を基準にしています
